<a href="https://colab.research.google.com/github/dejairps85-afk/o-ultimo-refugio/blob/main/O_Ultimo_Refugio_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SOBREVIVÊNCIA: O ÚLTIMO REFÚGIO — Protótipo 3D
**Projeto:** survival 3D em terceira pessoa, desenvolvido para executar pelo navegador a partir do Google Colab.

### O que já funciona
- Mundo 3D com terreno, árvores, pedras, arbustos e céu dinâmico.
- Personagem controlável por **WASD** (ou setas), corrida com **Shift** e câmera orbital com mouse.
- Coleta de madeira, pedra e frutos; inventário e barra de sobrevivência.
- Zumbis que perseguem o jogador; ataque com **Espaço**.
- Ciclo simplificado de dia/noite e condição de derrota.

> Este é o primeiro protótipo jogável. O Colab hospeda os arquivos e o navegador renderiza o 3D usando Three.js. Não é necessário instalar nada no computador ou celular, mas a experiência de controle é melhor em computador.

In [ ]:
# 1) Preparar o ambiente do Colab
from pathlib import Path
from IPython.display import display, HTML
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
import os, time

BASE = Path("/content/ultimo_refugio")
BASE.mkdir(exist_ok=True)
print("Pasta do projeto:", BASE)

Pasta do projeto: /content/ultimo_refugio


In [ ]:
html = r'''<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=no">
<title>O Último Refúgio — Survival 3D</title>
<style>
*{box-sizing:border-box}body{margin:0;overflow:hidden;background:#101820;color:#eef4ee;font-family:system-ui,Arial}
#hud{position:fixed;top:12px;left:12px;z-index:5;width:min(310px,calc(100vw - 24px));background:#101b20d9;border:1px solid #ffffff28;border-radius:14px;padding:12px;backdrop-filter:blur(8px)}
h1{font-size:15px;margin:0 0 8px;letter-spacing:.08em} .stat{font-size:12px;margin:6px 0 2px;display:flex;justify-content:space-between}
.track{height:8px;background:#ffffff20;border-radius:8px;overflow:hidden}.fill{height:100%;width:100%;transition:width .2s}
#inv{font-size:12px;margin-top:9px;line-height:1.7;color:#d7e3db}#hint{position:fixed;bottom:14px;left:50%;transform:translateX(-50%);z-index:5;background:#101b20d9;border:1px solid #ffffff28;border-radius:12px;padding:9px 13px;font-size:12px;text-align:center;max-width:95vw}
#message{position:fixed;top:18px;left:50%;transform:translateX(-50%);z-index:6;background:#7b332b;color:white;padding:10px 16px;border-radius:10px;display:none}
#cross{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);font-size:18px;color:#ffffff80;pointer-events:none}
#overlay{position:fixed;inset:0;z-index:10;background:#071016eF;display:flex;align-items:center;justify-content:center;text-align:center;padding:24px}
.panel{max-width:520px}.panel h2{font-size:clamp(28px,6vw,48px);margin:0 0 10px}.panel p{color:#c4d0ca;line-height:1.6}
button{border:0;border-radius:10px;padding:12px 20px;background:#c5e36b;color:#172018;font-weight:800;font-size:15px;cursor:pointer}
canvas{display:block}
</style>
</head>
<body>
<div id="hud">
<h1>O ÚLTIMO REFÚGIO <span style="color:#c5e36b">/ PROTÓTIPO 0.1</span></h1>
<div class="stat"><span>VIDA</span><b id="hpv">100</b></div><div class="track"><div id="hp" class="fill" style="background:#e45e55"></div></div>
<div class="stat"><span>FOME</span><b id="foodv">100</b></div><div class="track"><div id="food" class="fill" style="background:#e5b34f"></div></div>
<div class="stat"><span>SEDE</span><b id="waterv">100</b></div><div class="track"><div id="water" class="fill" style="background:#54b9df"></div></div>
<div id="inv">🪵 Madeira: 0 &nbsp; 🪨 Pedra: 0 &nbsp; 🍓 Frutos: 0</div>
<div id="clock" style="font-size:11px;color:#c5e36b;margin-top:5px">DIA 1 · 08:00</div>
</div>
<div id="message"></div><div id="cross">＋</div>
<div id="hint">WASD mover · Shift correr · Espaço atacar · E coletar · Mouse olhar · Arraste para girar câmera</div>
<div id="overlay"><div class="panel"><h2>O ÚLTIMO<br><span style="color:#c5e36b">REFÚGIO</span></h2>
<p>Você acordou em uma região isolada. Reúna recursos, mantenha-se alimentado e sobreviva aos infectados que aparecem quando a noite chega.</p>
<p><b>Objetivo:</b> sobreviva por 3 dias e prepare seu abrigo.</p>
<button id="start">ENTRAR NO MUNDO</button><p style="font-size:11px">Protótipo experimental · progresso não é salvo ainda</p></div></div>
<script type="module">
import * as THREE from 'https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js';
const scene=new THREE.Scene(); scene.background=new THREE.Color(0x91b9c2); scene.fog=new THREE.Fog(0x91b9c2,35,115);
const camera=new THREE.PerspectiveCamera(65,innerWidth/innerHeight,.1,250);
const renderer=new THREE.WebGLRenderer({antialias:true});renderer.setSize(innerWidth,innerHeight);renderer.setPixelRatio(Math.min(devicePixelRatio,2));renderer.shadowMap.enabled=true;renderer.shadowMap.type=THREE.PCFSoftShadowMap;document.body.appendChild(renderer.domElement);
scene.add(new THREE.HemisphereLight(0xc9e5ff,0x33402b,1.7));
const sun=new THREE.DirectionalLight(0xffe2b2,2.2);sun.position.set(-25,45,20);sun.castShadow=true;sun.shadow.mapSize.set(2048,2048);scene.add(sun);
const mat=(c,r=.9)=>new THREE.MeshStandardMaterial({color:c,roughness:r});
const ground=new THREE.Mesh(new THREE.PlaneGeometry(220,220),mat(0x536b43));ground.rotation.x=-Math.PI/2;ground.receiveShadow=true;scene.add(ground);
const grid=new THREE.GridHelper(220,55,0x6c8058,0x61764e);grid.position.y=.015;grid.material.opacity=.12;grid.material.transparent=true;scene.add(grid);
const player=new THREE.Group();scene.add(player);
const body=new THREE.Mesh(new THREE.CapsuleGeometry(.42,.75,5,10),mat(0x344d68));body.position.y=.95;body.castShadow=true;player.add(body);
const head=new THREE.Mesh(new THREE.SphereGeometry(.28,16,12),mat(0xd3a77f));head.position.y=1.62;head.castShadow=true;player.add(head);
const pack=new THREE.Mesh(new THREE.BoxGeometry(.48,.58,.22),mat(0x76563a));pack.position.set(0,1.05,-.38);player.add(pack);
const resources=[]; const rand=(a,b)=>a+Math.random()*(b-a);
function tree(x,z){let g=new THREE.Group();let trunk=new THREE.Mesh(new THREE.CylinderGeometry(.18,.25,2,7),mat(0x60432d));trunk.position.y=1;trunk.castShadow=true;g.add(trunk);let crown=new THREE.Mesh(new THREE.ConeGeometry(rand(1.05,1.45),rand(2.1,2.8),7),mat(0x294d36));crown.position.y=2.8;crown.castShadow=true;g.add(crown);g.position.set(x,0,z);scene.add(g);resources.push({obj:g,type:'wood',amount:1,rad:1.6});}
function rock(x,z){let m=new THREE.Mesh(new THREE.DodecahedronGeometry(rand(.45,.75),0),mat(0x777d79));m.position.set(x,.4,z);m.scale.y=.7;m.castShadow=true;scene.add(m);resources.push({obj:m,type:'stone',amount:1,rad:1.3});}
function bush(x,z){let m=new THREE.Mesh(new THREE.SphereGeometry(.6,9,7),mat(0x668c42));m.position.set(x,.45,z);m.scale.set(1.2,.8,1);scene.add(m);resources.push({obj:m,type:'berries',amount:2,rad:1.4});}
for(let i=0;i<100;i++){let x=rand(-85,85),z=rand(-85,85);if(Math.hypot(x,z)<9)continue;let q=Math.random();q<.55?tree(x,z):q<.78?rock(x,z):bush(x,z);}
const keys={};addEventListener('keydown',e=>{keys[e.code]=true;if(['Space','ArrowUp','ArrowDown','ArrowLeft','ArrowRight'].includes(e.code))e.preventDefault();});addEventListener('keyup',e=>keys[e.code]=false);
let yaw=0,camDist=8,drag=false,lastX=0,lastY=0,pitch=.24;
renderer.domElement.addEventListener('pointerdown',e=>{drag=true;lastX=e.clientX;lastY=e.clientY;renderer.domElement.setPointerCapture(e.pointerId)});
renderer.domElement.addEventListener('pointerup',()=>drag=false);
renderer.domElement.addEventListener('pointermove',e=>{if(!drag)return;yaw-=(e.clientX-lastX)*.006;pitch=Math.max(-.05,Math.min(.75,pitch+(e.clientY-lastY)*.003));lastX=e.clientX;lastY=e.clientY;});
const inv={wood:0,stone:0,berries:0};let hp=100,food=100,water=100,elapsed=0,started=false,dead=false,day=1;
const zombies=[];
function zombie(x,z){const g=new THREE.Group();const b=new THREE.Mesh(new THREE.CapsuleGeometry(.42,.7,4,8),mat(0x66806b));b.position.y=.9;b.castShadow=true;g.add(b);const h=new THREE.Mesh(new THREE.SphereGeometry(.27,12,10),mat(0x9caa83));h.position.y=1.55;g.add(h);g.position.set(x,0,z);scene.add(g);zombies.push({obj:g,hp:3,hit:0});}
for(let i=0;i<8;i++)zombie(rand(-55,55),rand(-55,55));
const msg=(s)=>{let e=document.getElementById('message');e.textContent=s;e.style.display='block';clearTimeout(msg.t);msg.t=setTimeout(()=>e.style.display='none',1800)};
function updateHud(){for(let [id,v] of [['hp',hp],['food',food],['water',water]])document.getElementById(id).style.width=Math.max(0,v)+'%';document.getElementById('hpv').textContent=Math.ceil(hp);document.getElementById('foodv').textContent=Math.ceil(food);document.getElementById('waterv').textContent=Math.ceil(water);document.getElementById('inv').textContent=`🪵 Madeira: ${inv.wood}   🪨 Pedra: ${inv.stone}   🍓 Frutos: ${inv.berries}`;}
function collect(){let near=null,dist=3;for(const r of resources){if(!r.obj.visible)continue;let d=player.position.distanceTo(r.obj.position);if(d<dist){near=r;dist=d;}}if(near){inv[near.type]+=near.amount;near.obj.visible=false;msg(near.type==='wood'?'Madeira coletada':near.type==='stone'?'Pedra coletada':'Frutos coletados');}}
function attack(){let target=null,dmin=3;for(const z of zombies){let d=player.position.distanceTo(z.obj.position);if(d<dmin){target=z;dmin=d;}}if(target){target.hp--;target.obj.position.add(new THREE.Vector3().subVectors(target.obj.position,player.position).normalize().multiplyScalar(1.3));if(target.hp<=0){scene.remove(target.obj);zombies.splice(zombies.indexOf(target),1);msg('Infectado eliminado');}else msg('Você atingiu o infectado!');}}
let lastCollect=false,lastAttack=false;
function gameOver(win=false){dead=true;document.querySelector('#overlay').style.display='flex';document.querySelector('.panel').innerHTML=win?'<h2>VOCÊ SOBREVIVEU!</h2><p>Três dias no mundo hostil. Este é apenas o começo da sua jornada.</p><button onclick="location.reload()">JOGAR NOVAMENTE</button>':'<h2>FIM DA JORNADA</h2><p>Você não conseguiu sobreviver. Reúna recursos e tente novamente.</p><button onclick="location.reload()">TENTAR NOVAMENTE</button>';}
document.getElementById('start').onclick=()=>{started=true;document.getElementById('overlay').style.display='none';};
const clockEl=document.getElementById('clock');let last=performance.now();
function animate(now){requestAnimationFrame(animate);let dt=Math.min((now-last)/1000,.05);last=now;
if(started&&!dead){
elapsed+=dt;food=Math.max(0,food-dt*.16);water=Math.max(0,water-dt*.23);if(food===0||water===0)hp-=dt*.9;
let forward=new THREE.Vector3(-Math.sin(yaw),0,-Math.cos(yaw)),right=new THREE.Vector3(Math.cos(yaw),0,-Math.sin(yaw)),move=new THREE.Vector3();
if(keys.KeyW||keys.ArrowUp)move.add(forward);if(keys.KeyS||keys.ArrowDown)move.sub(forward);if(keys.KeyD||keys.ArrowRight)move.add(right);if(keys.KeyA||keys.ArrowLeft)move.sub(right);
if(move.lengthSq()){move.normalize().multiplyScalar((keys.ShiftLeft||keys.ShiftRight?8:4.5)*dt);player.position.add(move);player.rotation.y=Math.atan2(-move.x,-move.z);}
if(keys.KeyE&&!lastCollect)collect();lastCollect=!!keys.KeyE;
if(keys.Space&&!lastAttack)attack();lastAttack=!!keys.Space;
if(keys.KeyF&&inv.berries>0){inv.berries--;food=Math.min(100,food+28);msg('Você comeu frutos');keys.KeyF=false;}
if(keys.KeyR){water=Math.min(100,water+35);keys.KeyR=false;msg('Você encontrou água da chuva');}
for(const z of zombies){let dir=new THREE.Vector3().subVectors(player.position,z.obj.position);let d=dir.length();if(d>1.35){dir.normalize();z.obj.position.add(dir.multiplyScalar(dt*(elapsed%120>70?1.7:.85)));z.obj.rotation.y=Math.atan2(-dir.x,-dir.z);}else{z.hit+=dt;if(z.hit>1.1){hp-=7;z.hit=0;}}}
let t=(elapsed%150)/150;let night=t>.67;sun.intensity=night?.25:2.2;scene.background.set(night?0x111b31:0x91b9c2);scene.fog.color.set(night?0x111b31:0x91b9c2);scene.fog.near=night?15:35;scene.fog.far=night?65:115;
day=Math.floor(elapsed/150)+1;let hh=Math.floor((8+(elapsed%150)/150*24)%24);clockEl.textContent=`DIA ${day} · ${String(hh).padStart(2,'0')}:00${night?' · NOITE':''}`;
if(elapsed>450)gameOver(true);if(hp<=0)gameOver(false);updateHud();
}
let target=new THREE.Vector3(player.position.x,1,player.position.z);let offset=new THREE.Vector3(Math.sin(yaw)*camDist,3+pitch*5,Math.cos(yaw)*camDist);camera.position.lerp(target.clone().add(offset),.12);camera.lookAt(target);
renderer.render(scene,camera);}
requestAnimationFrame(animate);
addEventListener('resize',()=>{camera.aspect=innerWidth/innerHeight;camera.updateProjectionMatrix();renderer.setSize(innerWidth,innerHeight);});
</script></body></html>'''
(BASE / 'index.html').write_text(html, encoding='utf-8')
print('Jogo 3D criado:', BASE / 'index.html')

Jogo 3D criado: /content/ultimo_refugio/index.html


In [ ]:

from IPython.display import display, HTML
from google.colab import output

# Obter endereço do servidor que já está funcionando
url = output.eval_js(
    f"google.colab.kernel.proxyPort({PORTA})"
)

# Exibir explicitamente o jogo em uma janela incorporada
display(HTML(f"""
<div style="width:100%; background:#111; padding:5px;
border-radius:12px;">
    <h3 style="color:#c5e36b;text-align:center;">
        O ÚLTIMO REFÚGIO 3D
    </h3>
    <iframe
        src="{url.rstrip('/')}/index.html"
        width="100%"
        height="650"
        style="border:0;border-radius:10px;background:#222;"
        allow="fullscreen; autoplay; gamepad"
        allowfullscreen>
    </iframe>
</div>
"""))


NameError: name 'PORTA' is not defined

In [ ]:

# DIAGNÓSTICO VISUAL DO JOGO 3D

from pathlib import Path

arquivo = BASE / "index.html"
html = arquivo.read_text(encoding="utf-8")

# Criar painel para mostrar erros diretamente na tela
painel = """
<div id="debugGame" style="
position:fixed;bottom:10px;left:10px;right:10px;
z-index:99999;background:#202b35;color:white;
padding:12px;border-radius:10px;font:13px monospace;
max-height:180px;overflow:auto">
DIAGNÓSTICO: carregando motor 3D...
</div>
<script>
function reportGameError(msg) {
  const box = document.getElementById('debugGame');
  if(box) box.innerHTML = '<b>ERRO NO JOGO:</b><br>' + msg;
}
window.addEventListener('error', e => reportGameError(e.message));
window.addEventListener('unhandledrejection', e =>
  reportGameError(String(e.reason))
);
</script>
"""

html = html.replace("<body>", "<body>" + painel, 1)

# Trocar importação estática por importação com captura de erro
html = html.replace(
    'import * as THREE from \'https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js\';',
    'try { const THREE = await import(\'https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js\');'
)

# Mostrar resultado da inicialização
html = html.replace(
    'requestAnimationFrame(animate);',
    'requestAnimationFrame(animate);',
    1
)

html = html.replace(
    '</script></body></html>',
    """
} catch(error) {
  reportGameError(error.stack || error.message || String(error));
}
setTimeout(() => {
  const box = document.getElementById('debugGame');
  if(box && box.textContent.includes('carregando')) {
    box.innerHTML = document.querySelector('canvas')
      ? 'MOTOR 3D CARREGADO. Verificando renderização...'
      : 'ERRO: O motor 3D não criou o canvas.';
  }
}, 6000);
</script></body></html>
"""
)

arquivo.write_text(html, encoding="utf-8")

print("Diagnóstico instalado!")
print("Atualize a janela do jogo para visualizar o resultado.")


Diagnóstico instalado!
Atualize a janela do jogo para visualizar o resultado.


In [ ]:

# TESTE INDEPENDENTE DO MOTOR 3D

from pathlib import Path
from IPython.display import display, HTML
from google.colab import output

teste = r"""
<!DOCTYPE html>
<html>
<head>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
body {
 background:#18252d;
 color:white;
 font-family:Arial;
 text-align:center;
 padding:20px;
}
canvas {display:block;margin:15px auto;max-width:100%}
</style>
</head>
<body>
<h2>TESTE GRÁFICO 3D</h2>
<div id="status">Verificando WebGL...</div>
<div id="scene"></div>

<script>
const status = document.getElementById('status');

if (!window.WebGLRenderingContext) {
 status.innerHTML = 'ERRO: WebGL não disponível neste navegador.';
} else {
 status.innerHTML = 'WebGL detectado. Carregando Three.js...';

 const script = document.createElement('script');
 script.src = 'https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js';

 script.onload = function() {
   try {
     const scene = new THREE.Scene();
     scene.background = new THREE.Color(0x263c49);

     const camera = new THREE.PerspectiveCamera(
       70, 1, 0.1, 100
     );
     camera.position.z = 4;

     const renderer = new THREE.WebGLRenderer({
       antialias:true
     });
     renderer.setSize(300,300);

     document.getElementById('scene').appendChild(
       renderer.domElement
     );

     const geometry = new THREE.BoxGeometry();
     const material = new THREE.MeshNormalMaterial();
     const cube = new THREE.Mesh(geometry, material);
     scene.add(cube);

     status.innerHTML = 'SUCESSO! MOTOR 3D FUNCIONANDO.';

     function animate() {
       requestAnimationFrame(animate);
       cube.rotation.x += 0.01;
       cube.rotation.y += 0.015;
       renderer.render(scene,camera);
     }
     animate();

   } catch(e) {
     status.innerHTML = 'ERRO 3D: ' + e.message;
   }
 };

 script.onerror = function() {
   status.innerHTML = 'ERRO: Não foi possível baixar Three.js.';
 };

 document.head.appendChild(script);
}
</script>
</body>
</html>
"""

(BASE / "diagnostico.html").write_text(teste, encoding="utf-8")

# Reutilizar o servidor existente
porta_teste = server.server_address[1]
endereco = output.eval_js(
    f"google.colab.kernel.proxyPort({porta_teste})"
)

display(HTML(f"""
<iframe
src="{endereco.rstrip('/')}/diagnostico.html"
width="100%"
height="480"
style="border:0;background:#18252d;border-radius:12px">
</iframe>
"""))


NameError: name 'server' is not defined

In [ ]:
!pwd

/content


In [ ]:
!ls -la

total 20
drwxr-xr-x 1 root root 4096 Nov 25 20:42 .
drwxr-xr-x 1 root root 4096 Nov 25 20:28 ..
drwxr-xr-x 4 root root 4096 Nov 18 15:58 .config
drwxr-xr-x 1 root root 4096 Nov 18 15:58 sample_data
drwxr-xr-x 2 root root 4096 Nov 25 20:42 ultimo_refugio


In [ ]:
!ls -lah /content/ultimo_refugio

total 28K
drwxr-xr-x 2 root root 4.0K Nov 25 20:42 .
drwxr-xr-x 1 root root 4.0K Nov 25 20:42 ..
-rw-r--r-- 1 root root 1.9K Nov 25 20:42 diagnostico.html
-rw-r--r-- 1 root root  13K Nov 25 20:42 index.html


In [ ]:
!grep -nEi "script|stylesheet|src=|href=|three|importmap" /content/ultimo_refugio/index.html

28:<script>
37:</script>
53:<script type="module">
54:try { const THREE = await import('https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js');
55:const scene=new THREE.Scene(); scene.background=new THREE.Color(0x91b9c2); scene.fog=new THREE.Fog(0x91b9c2,35,115);
56:const camera=new THREE.PerspectiveCamera(65,innerWidth/innerHeight,.1,250);
57:const renderer=new THREE.WebGLRenderer({antialias:true});renderer.setSize(innerWidth,innerHeight);renderer.setPixelRatio(Math.min(devicePixelRatio,2));renderer.shadowMap.enabled=true;renderer.shadowMap.type=THREE.PCFSoftShadowMap;document.body.appendChild(renderer.domElement);
58:scene.add(new THREE.HemisphereLight(0xc9e5ff,0x33402b,1.7));
59:const sun=new THREE.DirectionalLight(0xffe2b2,2.2);sun.position.set(-25,45,20);sun.castShadow=true;sun.shadow.mapSize.set(2048,2048);scene.add(sun);
60:const mat=(c,r=.9)=>new THREE.MeshStandardMaterial({color:c,roughness:r});
61:const ground=new THREE.Mesh(new THREE.PlaneGeometry(220,220),mat(0x

In [ ]:
!sed -n '45,125p' /content/ultimo_refugio/index.html

<div id="clock" style="font-size:11px;color:#c5e36b;margin-top:5px">DIA 1 · 08:00</div>
</div>
<div id="message"></div><div id="cross">＋</div>
<div id="hint">WASD mover · Shift correr · Espaço atacar · E coletar · Mouse olhar · Arraste para girar câmera</div>
<div id="overlay"><div class="panel"><h2>O ÚLTIMO<br><span style="color:#c5e36b">REFÚGIO</span></h2>
<p>Você acordou em uma região isolada. Reúna recursos, mantenha-se alimentado e sobreviva aos infectados que aparecem quando a noite chega.</p>
<p><b>Objetivo:</b> sobreviva por 3 dias e prepare seu abrigo.</p>
<button id="start">ENTRAR NO MUNDO</button><p style="font-size:11px">Protótipo experimental · progresso não é salvo ainda</p></div></div>
<script type="module">
try { const THREE = await import('https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js');
const scene=new THREE.Scene(); scene.background=new THREE.Color(0x91b9c2); scene.fog=new THREE.Fog(0x91b9c2,35,115);
const camera=new THREE.PerspectiveCamera(65,inne

In [ ]:
!grep -n "reportGameError" /content/ultimo_refugio/index.html

29:function reportGameError(msg) {
33:window.addEventListener('error', e => reportGameError(e.message));
35:  reportGameError(String(e.reason))
110:  reportGameError(error.stack || error.message || String(error));


In [ ]:
%cd /content/ultimo_refugio
!git init
!git add index.html diagnostico.html
!git config user.name "Dejair"
!git config user.email "SEU_EMAIL_DO_GITHUB"
!git commit -m "Primeira versão do O Último Refúgio 3D"

/content/ultimo_refugio
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/ultimo_refugio/.git/
[master (root-commit) 3fe7c8c] Primeira versão do O Último Refúgio 3D
 2 files changed, 200 insertions(+)
 create mode 100644 diagnostico.html
 create mode 100644 index.html


In [ ]:
%cd /content/ultimo_refugio
!git branch -M main
!git remote remove origin 2>/dev/null || true
!git remote add origin https://github.com/dejairps85-afk/o-ultimo-refugio.git
!git remote -v

/content/ultimo_refugio
origin	https://github.com/dejairps85-afk/o-ultimo-refugio.git (fetch)
origin	https://github.com/dejairps85-afk/o-ultimo-refugio.git (push)


In [ ]:
%cd /content/ultimo_refugio
!git push -u origin main

/content/ultimo_refugio
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
import getpass
import subprocess

token = getpass.getpass("Cole seu token GitHub: ")

remote = "https://github.com/dejairps85-afk/o-ultimo-refugio.git"

subprocess.run(
    ["git", "remote", "set-url", "origin",
     remote.replace("https://", f"https://{token}@")],
    check=True
)

subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    check=True
)

KeyboardInterrupt: Interrupted by user

In [ ]:
import getpass
import subprocess
import os
import base64

token = getpass.getpass("Novo token GitHub: ")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    env=env,
    check=True
)

token = None
auth = None

print("Publicação concluída!")

KeyboardInterrupt: Interrupted by user

In [ ]:
import subprocess

print("Verificando projeto...")

print(subprocess.run(
    ["git", "status", "--short"],
    capture_output=True, text=True
).stdout)

print("Verificando branch...")

print(subprocess.run(
    ["git", "branch", "--show-current"],
    capture_output=True, text=True
).stdout)

print("Diagnóstico concluído.")

Verificando projeto...

Verificando branch...
main

Diagnóstico concluído.


In [ ]:
import subprocess

r = subprocess.run(
    ["git", "remote", "-v"],
    capture_output=True,
    text=True
)

print(r.stdout)
print(r.stderr)

origin	https://github.com/dejairps85-afk/o-ultimo-refugio.git (fetch)
origin	https://github.com/dejairps85-afk/o-ultimo-refugio.git (push)




In [ ]:
import subprocess
import getpass
import os
import base64

token = getpass.getpass("Digite seu novo token GitHub: ")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

r = subprocess.run(
    ["git", "ls-remote", "origin"],
    env=env,
    capture_output=True,
    text=True,
    timeout=30
)

print("Código:", r.returncode)
print("Resultado:", r.stdout[:2000])
print("Erro:", r.stderr[:1000])

token = None
auth = None

KeyboardInterrupt: Interrupted by user

In [ ]:
import subprocess

try:
    r = subprocess.run(
        ["git", "ls-remote", "origin"],
        capture_output=True,
        text=True,
        timeout=15
    )

    print("Código:", r.returncode)
    print("Saída:", r.stdout[:1000])
    print("Erro:", r.stderr[:1000])

except subprocess.TimeoutExpired:
    print("TIMEOUT: conexão com GitHub demorou mais de 15 segundos.")

Código: 0
Saída: 
Erro: 


In [ ]:
import subprocess

r = subprocess.run(
    ["git", "log", "-1", "--oneline"],
    capture_output=True,
    text=True
)

print(r.stdout)
print(r.stderr)

3fe7c8c Primeira versão do O Último Refúgio 3D




In [ ]:
import subprocess

r = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

print(r.stdout)

On branch main
nothing to commit, working tree clean



In [ ]:
import subprocess

for comando in [
    ["git", "log", "-1", "--oneline"],
    ["git", "rev-parse", "HEAD"],
    ["git", "ls-remote", "origin", "refs/heads/main"]
]:
    r = subprocess.run(
        comando,
        capture_output=True,
        text=True,
        timeout=20
    )
    print("$", " ".join(comando))
    print(r.stdout or "(sem resultado)")
    print("Erro:", r.stderr or "nenhum")
    print("---")

$ git log -1 --oneline
3fe7c8c Primeira versão do O Último Refúgio 3D

Erro: nenhum
---
$ git rev-parse HEAD
3fe7c8cecad96829fbe9287f580578ad21bca85e

Erro: nenhum
---
$ git ls-remote origin refs/heads/main
(sem resultado)
Erro: nenhum
---


In [ ]:
import subprocess
import getpass
import os
import base64

token = getpass.getpass("Novo token GitHub: ")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

try:
    r = subprocess.run(
        ["git", "push", "-u", "origin", "main"],
        env=env,
        capture_output=True,
        text=True,
        timeout=60
    )

    print("Código:", r.returncode)
    print("Saída:", r.stdout)
    print("Erro:", r.stderr)

except subprocess.TimeoutExpired:
    print("ENVIO INTERROMPIDO: limite de 60 segundos.")

finally:
    token = None
    auth = None
    env = None

KeyboardInterrupt: Interrupted by user

In [ ]:
import subprocess
import os

os.chdir("/content/ultimo_refugio")

r = subprocess.run(
    ["git", "ls-files"],
    capture_output=True,
    text=True,
    timeout=15
)

arquivos = r.stdout.splitlines()

print("ARQUIVOS DO JOGO:", len(arquivos))

for arquivo in arquivos:
    tamanho = os.path.getsize(arquivo)
    print(f"{arquivo} — {tamanho / 1024:.1f} KB")

ARQUIVOS DO JOGO: 2
diagnostico.html — 1.8 KB
index.html — 12.1 KB


In [ ]:
import requests

url = "https://api.github.com/repos/dejairps85-afk/o-ultimo-refugio"

r = requests.get(url, timeout=15)

print("Status:", r.status_code)

if r.status_code == 200:
    print("Repositório:", r.json()["full_name"])
    print("Visibilidade:", r.json()["visibility"])
elif r.status_code == 404:
    print("Repositório não encontrado ou privado.")
else:
    print(r.text[:500])

Status: 200
Repositório: dejairps85-afk/o-ultimo-refugio
Visibilidade: public


In [ ]:
import requests
import getpass
import os

TOKEN = getpass.getpass("Cole seu novo token GitHub: ")

BASE = "https://api.github.com/repos/dejairps85-afk/o-ultimo-refugio"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

os.chdir("/content/ultimo_refugio")

arquivos = ["index.html", "diagnostico.html"]

for arquivo in arquivos:
    print(f"\nPublicando {arquivo}...")

    caminho = f"{BASE}/contents/{arquivo}"

    with open(arquivo, "rb") as f:
        conteudo = f.read()

    consulta = requests.get(
        caminho,
        headers=headers,
        timeout=20
    )

    dados = {
        "message": f"Publicação de {arquivo}",
        "content": __import__("base64").b64encode(
            conteudo
        ).decode()
    }

    if consulta.status_code == 200:
        dados["sha"] = consulta.json()["sha"]
    elif consulta.status_code != 404:
        print("Erro na consulta:", consulta.status_code)
        print(consulta.text[:500])
        break

    resposta = requests.put(
        caminho,
        headers=headers,
        json=dados,
        timeout=30
    )

    print("Status:", resposta.status_code)

    if resposta.status_code in (200, 201):
        print("Publicado:", arquivo)
    else:
        print("Erro:", resposta.text[:1000])
        break

TOKEN = None
headers = None

KeyboardInterrupt: Interrupted by user

## Controles e próximos marcos

| Ação | Controle |
|---|---|
| Andar | W A S D / setas |
| Correr | Shift |
| Coletar recurso próximo | E |
| Atacar infectado próximo | Espaço |
| Comer frutos coletados | F |
| Recuperar água (ação provisória) | R |
| Girar câmera | Arrastar com mouse |

### Próxima versão planejada
1. Abrigo construível com sistema de receitas (madeira + pedra).
2. Inventário visual, ferramentas, armas e durabilidade.
3. Água como recurso real: rios, poços e purificação.
4. Salvamento de progresso.
5. Mapa maior dividido em biomas, missões e chefes.
6. Áudio, animações e otimização para celular.

**Nota técnica:** o Google Colab é um ambiente de prototipagem, não um servidor de produção. Para publicar o jogo e manter partidas online, depois migraremos o frontend para hospedagem web adequada e adicionaremos backend/salvamento.